# Using existing trajectories with NanoVer

This notebook demonstrates how load a trajectory file with MDAnalysis and combine it with NanoVer to run as interactive molecular dynamics (iMD) simulation that can be viewed in the NanoVer iMD-XR client.

## Trajectory setup

First we set up an MDAnalysis universe from topology and a short trajectory of oseltamivir unbinding from neuraminidase.

In [1]:
TOPOLOGY_PATH = "../systems/3TI6_ose_wt.pdb"
TRAJECTORY_PATH = "../systems/ose_wt.dcd"

In [2]:
from MDAnalysis import Universe

universe = Universe(
    TOPOLOGY_PATH,
    TRAJECTORY_PATH,
    trajectory=True,
    to_guess=("bonds",),
)
universe.trajectory

C:\Users\ragzo\Documents\REPOS\nanover-server-py-uv\.venv\Lib\site-packages\MDAnalysis\coordinates\DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<DCDReader ../systems/ose_wt.dcd with 24 frames of 5877 atoms>

## NanoVer playback and server setup

Next we wrap the MDAnalysis universe in a NanoVer simulation (this provides compatibility with NanoVer), and start a server providing access to the simulation over the network.

In [3]:
from nanover.mdanalysis import UniverseSimulation

traj_sim = UniverseSimulation.from_universe(universe)
traj_sim.playback_factor = 5

In [4]:
from nanover.app import OmniRunner

imd_runner = OmniRunner.with_basic_server(traj_sim, port=0, name="trajectory playback example")
imd_runner.print_basic_info()
imd_runner.load(0)

Serving "trajectory playback example" (ws://localhost:64104), discoverable on all interfaces on port 54545
Available simulations:
[0]: "../systems/3TI6_ose_wt.pdb"
Switched to [0]: "../systems/3TI6_ose_wt.pdb"
Switched to [0]: "../systems/3TI6_ose_wt.pdb"


Before we look at the playback live, we'll use a utility to tell the visualisation clients to show to the protein as a cartoon and the ligand as liquorice:

In [7]:
from nanover.jupyter import NanoverJupyterUtilities

indices = universe.select_atoms("resname OSE").indices

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.selections.update_selection("root", renderer="cartoon")
utilities.selections.update_selection("ligand", renderer="liquorice", particle_ids=indices)

## Interacting in virtual reality

Finally, we connect using the [NanoVer iMD-XR client](https://irl2.github.io/nanover-docs/installation.html#installing-the-imd-xr-client) to see the looping trajectory.

<video src="../figures/trajectory-playback-example.webm" controls>